# Diagnostics walkthrough

This notebook demonstrates `ballpushing_utils.diagnostics` against a
**stub Fly** — no SLEAP data required. It mirrors the hermetic unit
tests under `tests/unit/diagnostics/` and is designed to be the first
thing to run after `pip install -e ".[dev,interactive]"` to confirm
the diagnostics surface is intact.

For a real fly on the workstation, replace the stub construction
cell with:

```python
from ballpushing_utils import Fly, dataset
fly = Fly(dataset("path/to/fly"), as_individual=True)
```

Everything below is unchanged.

## 1. Build a stub Fly

The diagnostics builders only touch a thin slice of the `Fly`
surface: `experiment.fps`, `tracking_data.interaction_events`,
`event_summaries`, and the `metadata.name`. We fake those with plain
dataclasses so the rest of the notebook runs offline.

In [ ]:
from dataclasses import dataclass, field
from types import SimpleNamespace
from typing import Any


@dataclass
class StubExperiment:
    fps: float = 30.0


@dataclass
class StubTrackingData:
    interaction_events: dict | None = None


@dataclass
class StubFly:
    experiment: StubExperiment = field(default_factory=StubExperiment)
    tracking_data: StubTrackingData = field(default_factory=StubTrackingData)
    event_summaries: dict[str, dict[str, Any]] | None = None
    metadata: SimpleNamespace = field(
        default_factory=lambda: SimpleNamespace(name="demo_fly")
    )
    directory: str = "/tmp/demo_fly"


fly = StubFly()
fly.tracking_data.interaction_events = {
    0: {
        0: [
            # [start_frame, end_frame, displacement_px]
            [30, 90, 2.5],       # minor  (< signif_threshold 5 px)
            [300, 360, 12.0],    # significant
            [900, 1020, 35.0],   # major  (>= major_threshold 20 px)
            [1500, 1560, 7.5],   # significant
        ],
        1: [
            [600, 660, 8.0],
        ],
    }
}
fly.event_summaries = {
    "fly_0_ball_0": {
        "fly_idx": 0,
        "ball_idx": 0,
        "nb_events": 12,                        # ok
        "significant_ratio": 0.4,               # ok
        "max_distance": 250.0,                  # ok
        "chamber_ratio": 1.5,                   # high (>1.0)
        "median_pause_duration": float("nan"),  # nan
        "made_up_metric": 7.0,                  # not in default ranges
    },
    "fly_0_ball_1": {
        "fly_idx": 0,
        "ball_idx": 1,
        "nb_events": -3,                        # low (<0)
        "significant_ratio": 0.0,
    },
}
fly

## 2. Event timeline

`build_event_timeline` returns a `DiagnosticReport` with two
sections: a per-event table (one row per detected interaction,
including `start_mm_ss` / `end_mm_ss` for video scrubbing) and a
Gantt-style figure.

In [ ]:
from ballpushing_utils.diagnostics import build_event_timeline

timeline = build_event_timeline(fly)
for s in timeline.sections:
    print(f"- {s.name}: {s.description[:80]}")
print("metadata:", timeline.metadata)

In [ ]:
events_section = next(s for s in timeline.sections if s.name == "events")
events_section.table

In [ ]:
timeline_section = next(s for s in timeline.sections if s.name == "timeline")
timeline_section.figure

### Tweak thresholds

Drop the significance threshold below the smallest displacement to
see every event re-classified. This is useful when debugging
`interaction_threshold` / `signif_event_threshold` in `Config`.

In [ ]:
relaxed = build_event_timeline(fly, signif_threshold=1.0, major_threshold=10.0)
events_relaxed = next(s for s in relaxed.sections if s.name == "events")
events_relaxed.table[[
    "event_idx", "ball_displacement_px", "classification",
    "is_significant", "is_major",
]]

## 3. Metric report

`build_metric_report` walks `fly.event_summaries` and flags each
metric as `ok` / `low` / `high` / `nan` using
`DEFAULT_METRIC_RANGES`. Metrics with no declared range still
appear in the table with `verdict="ok"` — they are never silently
dropped.

In [ ]:
from ballpushing_utils.diagnostics import build_metric_report

metrics = build_metric_report(fly)
for s in metrics.sections:
    print(f"- {s.name}")
print("metadata:", metrics.metadata)

In [ ]:
next(s for s in metrics.sections if s.name == "metrics").table

In [ ]:
summary_section = next(
    (s for s in metrics.sections if s.name == "summary"), None
)
summary_section.table if summary_section is not None else "No summary section"

## 4. Persist to disk

`write_report` materialises a report into a per-run folder:

```
<output_dir>/
    summary.md        # human-readable overview (20-row table previews)
    events.csv        # full event table
    metrics.csv       # full metric table
    plots/            # figure PNGs
```

In [ ]:
import tempfile
from pathlib import Path

from ballpushing_utils.diagnostics import write_report

with tempfile.TemporaryDirectory() as td:
    out = write_report(timeline, Path(td) / "timeline_demo")
    print("Summary written to:", out)
    print("Files:", sorted(p.relative_to(out.parent).as_posix() for p in out.parent.rglob("*")))
    print()
    print(out.read_text()[:2000])

## 5. Where to go next

- For an interactive browser-based dashboard over a real fly,
  run `python tools/diagnostics_dashboard.py <fly_directory>`.
- For batch diagnostics on every fly in an experiment, use
  `build_event_timelines_for_experiment(exp)` and pipe each
  `DiagnosticReport` into `write_report`.
- To add a new range check, extend
  `ballpushing_utils.diagnostics.metric_report.DEFAULT_METRIC_RANGES`
  and the corresponding unit test under
  `tests/unit/diagnostics/test_metric_report.py`.